I'n ylYesthere

In [ ]:
C

# WxWords — Cloud Classification Training

Train a ground-based cloud classifier on the CCSN dataset using GPU.

**Before running:** Go to Runtime → Change runtime type → **T4 GPU**

## 1. Setup & Install

In [ ]:
!pip install -q tensorflowjs

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Download CCSN Dataset

**Two options — pick one:**

- **Option A (easiest):** If you already have the dataset locally (e.g. from `wxwords/data/CCSN_v2/`), zip it and upload when prompted.
- **Option B:** Use Kaggle (free account, no payment). Get your API key from https://www.kaggle.com/settings → "Create New Token" → downloads `kaggle.json`. Upload it when prompted below.

In [ ]:
import os
from pathlib import Path
from google.colab import files

DATA_DIR = Path("data/CCSN_v2")

if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    print("Dataset already present, skipping download.")
else:
    print("Choose how to get the dataset:")
    print("  1 - Upload a zip of CCSN_v2/ from your local machine")
    print("  2 - Download from Kaggle (upload kaggle.json)")
    choice = input("\nEnter 1 or 2: ").strip()

    if choice == "1":
        # Option A: Upload zip
        print("\nUpload your CCSN_v2.zip (zip the CCSN_v2 folder):")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        os.makedirs("data", exist_ok=True)
        import shutil
        shutil.move(zip_name, f"data/{zip_name}")
        import zipfile
        with zipfile.ZipFile(f"data/{zip_name}", "r") as z:
            z.extractall("data/")
        # Handle nested dirs - find CCSN_v2 wherever it landed
        if not DATA_DIR.exists():
            for p in Path("data").rglob("CCSN_v2"):
                if p.is_dir():
                    shutil.move(str(p), str(DATA_DIR))
                    break
        print(f"Extracted to {DATA_DIR}")

    else:
        # Option B: Kaggle
        os.system("pip install -q kaggle")
        if not Path("/root/.kaggle/kaggle.json").exists():
            print("\nUpload your kaggle.json:")
            uploaded = files.upload()
            os.makedirs("/root/.kaggle", exist_ok=True)
            with open("/root/.kaggle/kaggle.json", "wb") as f:
                f.write(uploaded["kaggle.json"])
            os.chmod("/root/.kaggle/kaggle.json", 0o600)
        os.system(
            "kaggle datasets download "
            "-d mmichelli/cirrus-cumulus-stratus-nimbus-ccsn-database "
            "-p data --unzip"
        )

# Verify
classes = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
total = sum(len(list((DATA_DIR / c).glob("*"))) for c in classes)
print(f"\nDataset: {total} images across {len(classes)} classes: {classes}")

## 3. Prepare Train/Val Split

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data/CCSN_v2")
SPLIT_DIR = Path("data/split")

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for cls_dir in sorted(DATA_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    images = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.jpeg")) + list(cls_dir.glob("*.png"))
    train_imgs, val_imgs = train_test_split(images, test_size=0.2, random_state=42)

    for split, img_list in [("train", train_imgs), ("val", val_imgs)]:
        dest = SPLIT_DIR / split / cls_dir.name
        dest.mkdir(parents=True, exist_ok=True)
        for img in img_list:
            shutil.copy2(img, dest / img.name)

    print(f"{cls_dir.name}: {len(train_imgs)} train, {len(val_imgs)} val")

print("\nSplit complete!")

## 4. Compare Backbones (ResNet50V2, EfficientNetB0, MobileNetV2)

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BACKBONES = {
    "resnet50v2": {
        "base_fn": keras.applications.ResNet50V2,
        "preprocess": keras.applications.resnet_v2.preprocess_input,
        "unfreeze": 50,
    },
    "efficientnet": {
        "base_fn": keras.applications.EfficientNetB0,
        "preprocess": keras.applications.efficientnet.preprocess_input,
        "unfreeze": 40,
    },
    "mobilenetv2": {
        "base_fn": keras.applications.MobileNetV2,
        "preprocess": keras.applications.mobilenet_v2.preprocess_input,
        "unfreeze": 50,
    },
}


def train_backbone(name: str, epochs_p1: int = 20, epochs_p2: int = 40) -> float:
    print(f"\n{'='*60}")
    print(f"  Training: {name.upper()}")
    print(f"{'='*60}\n")

    config = BACKBONES[name]

    train_datagen = ImageDataGenerator(
        preprocessing_function=config["preprocess"],
        rotation_range=20,
        width_shift_range=0.15,
        height_shift_range=0.15,
        horizontal_flip=True,
        zoom_range=0.15,
        shear_range=0.1,
        brightness_range=(0.8, 1.2),
    )
    val_datagen = ImageDataGenerator(preprocessing_function=config["preprocess"])

    train_gen = train_datagen.flow_from_directory(
        SPLIT_DIR / "train", target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True,
    )
    val_gen = val_datagen.flow_from_directory(
        SPLIT_DIR / "val", target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False,
    )

    num_classes = len(train_gen.class_indices)
    class_weights = compute_class_weight(
        "balanced", classes=np.arange(num_classes), y=train_gen.classes
    )
    cw = dict(enumerate(class_weights))

    # Build model
    base = config["base_fn"](
        input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
    )
    base.trainable = False

    model = keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])

    # Phase 1: frozen base
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    model.fit(
        train_gen, validation_data=val_gen, epochs=epochs_p1,
        class_weight=cw,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        )],
    )
    _, p1_acc = model.evaluate(val_gen)
    print(f"  Phase 1 val acc: {p1_acc:.4f}")

    # Phase 2: fine-tune
    base.trainable = True
    for layer in base.layers[:-config["unfreeze"]]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="categorical_crossentropy", metrics=["accuracy"],
    )
    model.fit(
        train_gen, validation_data=val_gen, epochs=epochs_p2,
        class_weight=cw,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=8, restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6
            ),
        ],
    )

    _, final_acc = model.evaluate(val_gen)
    print(f"  FINAL val acc ({name}): {final_acc:.4f}")

    model.save(f"cloud_{name}.keras")
    return final_acc, model, list(train_gen.class_indices.keys())


# Run all 3
results = {}
best_model = None
best_acc = 0
class_names = None

for name in BACKBONES:
    acc, model, cn = train_backbone(name)
    results[name] = acc
    if acc > best_acc:
        best_acc = acc
        best_model = model
        best_name = name
        class_names = cn

print(f"\n{'='*60}")
print("  RESULTS")
print(f"{'='*60}")
for n, a in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {n:20s}: {a:.4f}")
print(f"\n  Best: {best_name} ({best_acc:.4f})")

## 5. Extended Training on Best Backbone

In [ ]:
print(f"Extended training on: {best_name} (starting from {best_acc:.4f})")

config = BACKBONES[best_name]

# Heavier augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=config["preprocess"],
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.15,
    brightness_range=(0.7, 1.3),
    fill_mode="reflect",
)
val_datagen = ImageDataGenerator(preprocessing_function=config["preprocess"])

train_gen = train_datagen.flow_from_directory(
    SPLIT_DIR / "train", target_size=IMG_SIZE,
    batch_size=16, class_mode="categorical", shuffle=True,
)
val_gen = val_datagen.flow_from_directory(
    SPLIT_DIR / "val", target_size=IMG_SIZE,
    batch_size=16, class_mode="categorical", shuffle=False,
)

cw = dict(enumerate(compute_class_weight(
    "balanced", classes=np.arange(len(train_gen.class_indices)), y=train_gen.classes
)))

# Unfreeze more layers
for layer in best_model.layers[0].layers:
    layer.trainable = True
for layer in best_model.layers[0].layers[:-100]:
    layer.trainable = False

# Cosine decay schedule
EPOCHS_EXT = 60
steps = len(train_gen) * EPOCHS_EXT
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-5, decay_steps=steps, alpha=1e-7
)

best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

history = best_model.fit(
    train_gen, validation_data=val_gen, epochs=EPOCHS_EXT,
    class_weight=cw,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=15, restore_best_weights=True
        ),
        keras.callbacks.ModelCheckpoint(
            "cloud_best.keras", monitor="val_accuracy",
            save_best_only=True, verbose=1
        ),
    ],
)

_, final_acc = best_model.evaluate(val_gen)
print(f"\nFINAL extended accuracy: {final_acc:.4f}")
best_model.save("cloud_best.keras")

## 6. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["accuracy"], label="Train")
ax1.plot(history.history["val_accuracy"], label="Val")
ax1.set_title("Accuracy")
ax1.legend()

ax2.plot(history.history["loss"], label="Train")
ax2.plot(history.history["val_loss"], label="Val")
ax2.set_title("Loss")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Convert to TensorFlow.js

In [ ]:
import json
import tensorflowjs as tfjs

# Convert best model to TF.js format
TFJS_DIR = "models_tfjs"
tfjs.converters.save_keras_model(best_model, TFJS_DIR)

# Save class names alongside
with open(f"{TFJS_DIR}/class_names.json", "w") as f:
    json.dump(class_names, f)

print(f"TF.js model saved to {TFJS_DIR}/")
!ls -lh {TFJS_DIR}/

## 8. Download Model

Downloads the TF.js model as a zip file. Extract into your `wxwords/models/tfjs/` directory.

In [ ]:
!zip -r cloud_model_tfjs.zip {TFJS_DIR}/

from google.colab import files
files.download("cloud_model_tfjs.zip")

print("\nDone! Extract into wxwords/models/tfjs/ and the website will pick it up.")

## 9. Quick Test on a Sample Image

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

CLOUD_INFO = {
    'Ac': 'Altocumulus', 'As': 'Altostratus', 'Cb': 'Cumulonimbus',
    'Cc': 'Cirrocumulus', 'Ci': 'Cirrus', 'Cs': 'Cirrostratus',
    'Ct': 'Contrail', 'Cu': 'Cumulus', 'Ns': 'Nimbostratus',
    'Sc': 'Stratocumulus', 'St': 'Stratus',
}

# Test on a random val image
import random
val_dir = SPLIT_DIR / "val"
random_class = random.choice(list(val_dir.iterdir()))
random_img = random.choice(list(random_class.glob("*")))

img = load_img(random_img, target_size=IMG_SIZE)
img_array = img_to_array(img)
img_preprocessed = config["preprocess"](img_array.copy())
img_batch = np.expand_dims(img_preprocessed, axis=0)

preds = best_model.predict(img_batch)[0]
top3 = np.argsort(preds)[-3:][::-1]

plt.figure(figsize=(6, 6))
plt.imshow(img_array.astype("uint8"))
plt.title(f"True: {random_class.name} ({CLOUD_INFO[random_class.name]})")
plt.axis("off")
plt.show()

print(f"\nTrue label: {random_class.name} ({CLOUD_INFO[random_class.name]})")
print(f"\nTop 3 predictions:")
for i in top3:
    code = class_names[i]
    print(f"  {code} ({CLOUD_INFO[code]}): {preds[i]*100:.1f}%")